# 02 | Kiểm định ngoài model V5 trên DeepBeat, PulseWatch, Liu và bộ PAC/PVC

Chạy model V5 đã đóng băng (`models/v5/healthsense_af_v5_rf_ac_frozen.pkl`) trên bốn nguồn dữ liệu không dùng để huấn luyện. Mỗi ô gọi một script trong `notebooks/v5/scripts/`; đặc trưng và dự đoán sinh ra ghi vào `results/v5/` (bị gitignore vì dẫn xuất từ dữ liệu Synapse).

Cần có `data/raw/` của DeepBeat, PulseWatch, Liu (xem `data/README.md`). Kết luận của bước này, PAC/PVC là lớp âm tính khó, là lý do chuyển sang V6; đặc trưng PulseWatch sinh ra ở đây cũng là đầu vào của `notebooks/v6/00_extract_features_and_splits`.

In [ ]:
import os, sys, time, subprocess, glob
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "src" / "healthsense_ml").exists():
    if ROOT.parent == ROOT: raise RuntimeError("Không tìm thấy gốc repo")
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT / "src"))
print("Repo root:", ROOT)

def run(script, *args):
    """Chạy một script từ gốc repo bằng đúng Python của kernel; in log; dừng nếu lỗi."""
    t0 = time.time()
    print(f"\n$ python {script} {' '.join(args)}", flush=True)
    r = subprocess.run([sys.executable, script, *args], cwd=ROOT, text=True, encoding="utf-8", errors="replace",
                       capture_output=True, env={**os.environ, "PYTHONIOENCODING": "utf-8", "PYTHONPATH": "src"})
    print(r.stdout[-4000:])
    if r.returncode:
        print(r.stderr[-4000:]); raise RuntimeError(f"{script} lỗi, mã {r.returncode}")
    print(f"  xong trong {time.time()-t0:.0f}s")

## 1. DeepBeat

In [ ]:
run("notebooks/v5/scripts/deepbeat_frozen_v5_external.py")

## 2. PulseWatch, gồm cửa sổ PAC/PVC

In [ ]:
run("notebooks/v5/scripts/pulsewatch_frozen_v5_external.py")

## 3. Liu 2022, cửa sổ giả 30 giây

In [ ]:
run("notebooks/v5/scripts/liu_pseudo30_frozen_v5.py")

## 4. Bộ thử PAC/PVC

In [ ]:
run("notebooks/v5/scripts/peakdet_pac_pvc_frozen_v5.py")

## 5. Bảng tóm tắt

In [ ]:
import pandas as pd
for f in sorted(glob.glob("results/v5/*_summary.csv")):
    print("\n==", f); display(pd.read_csv(f).round(4))